# Retail Sales & Profitability Analysis
### Major Python Libraries Project

**Objective:** Analyze retail transactions to understand revenue, profit, product performance, customer segments, regional trends, discounts, and shipping patterns.

**Libraries:** Pandas, NumPy, Matplotlib, Seaborn.

> **Dataset note:** This is a simulated educational dataset generated for this project. It is not real company sales data and should not be presented as actual business performance.

## 1. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Loading the Dataset

In [ ]:
df = pd.read_csv("../dataset/retail_sales_uncleaned.csv")
df.head()

## 3. Initial Dataset Exploration

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nDataset information:")
df.info()

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\\nDuplicate records:", df.duplicated().sum())

In [ ]:
df.describe(include="all").T

## 4. Data Cleaning

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

# Standardize text columns
text_cols = ["Customer_Segment", "Product_Category", "Product", "Region", "City", "Payment_Method"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip().str.title()

# Convert date and numeric columns
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
numeric_cols = [
    "Units_Sold", "Unit_Price", "Discount", "Sales_Amount",
    "Discount_Amount", "Net_Sales", "Cost", "Profit", "Shipping_Days"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing numeric values with median
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with mode
for col in text_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df = df.dropna(subset=["Order_Date"]).copy()
print("Shape after cleaning:", df.shape)
print("Missing values remaining:", df.isnull().sum().sum())
print("Duplicate rows remaining:", df.duplicated().sum())

## 5. Data Transformation and Feature Engineering

In [ ]:
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Quarter"] = df["Order_Date"].dt.quarter
df["Day_Name"] = df["Order_Date"].dt.day_name()

# Profit margin as a percentage of net sales
df["Profit_Margin_Percent"] = np.where(
    df["Net_Sales"] > 0,
    (df["Profit"] / df["Net_Sales"]) * 100,
    0
).round(2)

# Project-defined discount bands
df["Discount_Category"] = pd.cut(
    df["Discount"],
    bins=[-0.001, 0, 0.10, 0.20, 0.31],
    labels=["No Discount", "Low Discount", "Medium Discount", "High Discount"]
)

# Project-defined order value bands
df["Order_Value_Category"] = pd.cut(
    df["Net_Sales"],
    bins=[-1, 5000, 20000, 60000, np.inf],
    labels=["Low", "Medium", "High", "Very High"]
)

df = df.reset_index(drop=True)
df.head()

## 6. Data Quality Checks

In [ ]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Order date range:", df["Order_Date"].min(), "to", df["Order_Date"].max())
print("Categories:", sorted(df["Product_Category"].unique()))
print("Regions:", sorted(df["Region"].unique()))

## 7. Exploratory Data Analysis and Visualizations

### 7.1 Monthly Sales Revenue Trend

In [ ]:
monthly_sales = df.groupby(["Year", "Month"])["Net_Sales"].sum().reset_index()
monthly_sales["Period"] = monthly_sales["Year"].astype(str) + "-" + monthly_sales["Month"].astype(str).str.zfill(2)
sns.lineplot(data=monthly_sales, x="Period", y="Net_Sales", marker="o")
plt.title("Monthly Net Sales Revenue Trend")
plt.xlabel("Year-Month")
plt.ylabel("Net sales")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

**Observation:** This chart compares net sales across months and helps reveal broad sales trends.

### 7.2 Total Profit by Product Category

In [ ]:
category_profit = df.groupby("Product_Category")["Profit"].sum().sort_values(ascending=False)
sns.barplot(x=category_profit.index, y=category_profit.values, color="seagreen")
plt.title("Total Profit by Product Category")
plt.xlabel("Product category")
plt.ylabel("Total profit")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

**Observation:** Compare total profit generated by each product category.

### 7.3 Top 10 Products by Net Sales

In [ ]:
top_products = df.groupby("Product")["Net_Sales"].sum().nlargest(10).sort_values()
sns.barplot(x=top_products.values, y=top_products.index, color="cornflowerblue")
plt.title("Top 10 Products by Net Sales")
plt.xlabel("Net sales")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

**Observation:** This chart identifies products contributing the highest net sales in the dataset.

### 7.4 Sales by Region

In [ ]:
region_sales = df.groupby("Region")["Net_Sales"].sum().sort_values(ascending=False)
sns.barplot(x=region_sales.index, y=region_sales.values, color="darkorange")
plt.title("Net Sales by Region")
plt.xlabel("Region")
plt.ylabel("Net sales")
plt.tight_layout()
plt.show()

**Observation:** Compare regional net sales to understand how sales are distributed geographically.

### 7.5 Order Volume by Customer Segment

In [ ]:
segment_counts = df["Customer_Segment"].value_counts()
sns.barplot(x=segment_counts.index, y=segment_counts.values, color="mediumpurple")
plt.title("Number of Orders by Customer Segment")
plt.xlabel("Customer segment")
plt.ylabel("Number of orders")
plt.tight_layout()
plt.show()

**Observation:** This chart compares transaction counts across customer segments.

### 7.6 Discount Percentage vs Profit

In [ ]:
sns.scatterplot(data=df, x="Discount", y="Profit", hue="Product_Category", alpha=.65)
plt.title("Discount Percentage vs Profit")
plt.xlabel("Discount rate")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()

**Observation:** Explore how profit values vary across discount rates and product categories. This is descriptive and does not prove that discounts cause profit changes.

### 7.7 Distribution of Profit Margins

In [ ]:
sns.histplot(data=df, x="Profit_Margin_Percent", bins=30, kde=True, color="teal")
plt.title("Distribution of Profit Margin")
plt.xlabel("Profit margin (%)")
plt.ylabel("Number of orders")
plt.tight_layout()
plt.show()

**Observation:** The histogram shows the spread of profit margins across orders.

### 7.8 Payment Method Distribution

In [ ]:
payment_counts = df["Payment_Method"].value_counts()
plt.pie(payment_counts.values, labels=payment_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Orders by Payment Method")
plt.tight_layout()
plt.show()

**Observation:** This chart summarizes the share of orders using each payment method.

### 7.9 Monthly Profit Trend

In [ ]:
monthly_profit = df.groupby(["Year", "Month"])["Profit"].sum().reset_index()
monthly_profit["Period"] = monthly_profit["Year"].astype(str) + "-" + monthly_profit["Month"].astype(str).str.zfill(2)
sns.lineplot(data=monthly_profit, x="Period", y="Profit", marker="o", color="green")
plt.title("Monthly Profit Trend")
plt.xlabel("Year-Month")
plt.ylabel("Total profit")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

**Observation:** Compare profit across months and observe whether profit follows a similar pattern to sales.

### 7.10 Sales by Product Category

In [ ]:
category_sales = df.groupby("Product_Category")["Net_Sales"].sum().sort_values(ascending=False)
sns.barplot(x=category_sales.index, y=category_sales.values, color="steelblue")
plt.title("Net Sales by Product Category")
plt.xlabel("Product category")
plt.ylabel("Net sales")
plt.tight_layout()
plt.show()

**Observation:** Identify which product categories contribute the largest share of net sales.

### 7.11 Shipping Days Distribution

In [ ]:
sns.histplot(data=df, x="Shipping_Days", bins=12, discrete=True, color="slateblue")
plt.title("Distribution of Shipping Days")
plt.xlabel("Shipping days")
plt.ylabel("Number of orders")
plt.tight_layout()
plt.show()

**Observation:** The distribution shows how many orders fall into each shipping-duration value.

### 7.12 Average Profit Margin by Category

In [ ]:
margin_category = df.groupby("Product_Category")["Profit_Margin_Percent"].mean().sort_values(ascending=False)
sns.barplot(x=margin_category.index, y=margin_category.values, color="mediumseagreen")
plt.title("Average Profit Margin by Product Category")
plt.xlabel("Product category")
plt.ylabel("Average profit margin (%)")
plt.tight_layout()
plt.show()

**Observation:** Compare average margin percentages across categories rather than comparing only total profit.

### 7.13 Top 10 Customers by Net Sales

In [ ]:
top_customers = df.groupby("Customer_ID")["Net_Sales"].sum().nlargest(10).sort_values()
sns.barplot(x=top_customers.values, y=top_customers.index, color="coral")
plt.title("Top 10 Customers by Net Sales")
plt.xlabel("Net sales")
plt.ylabel("Customer ID")
plt.tight_layout()
plt.show()

**Observation:** This identifies the customers with the highest cumulative net sales in the simulated dataset.

### 7.14 Correlation Heatmap

In [ ]:
corr_cols = ["Units_Sold", "Unit_Price", "Discount", "Sales_Amount",
             "Discount_Amount", "Net_Sales", "Cost", "Profit",
             "Profit_Margin_Percent", "Shipping_Days"]
sns.heatmap(df[corr_cols].corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Sales and Profit Variables")
plt.tight_layout()
plt.show()

**Observation:** Correlation summarizes linear relationships among numeric variables. It does not establish causation.

### 7.15 Quarterly Sales Comparison

In [ ]:
quarter_sales = df.groupby(["Year", "Quarter"])["Net_Sales"].sum().reset_index()
quarter_sales["Period"] = quarter_sales["Year"].astype(str) + " Q" + quarter_sales["Quarter"].astype(str)
sns.barplot(data=quarter_sales, x="Period", y="Net_Sales", color="goldenrod")
plt.title("Quarterly Net Sales Comparison")
plt.xlabel("Year and quarter")
plt.ylabel("Net sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Observation:** Quarterly aggregation provides a higher-level view of sales performance.

## 8. Business Analysis and Key Metrics

In [ ]:
total_revenue = df["Sales_Amount"].sum()
total_discount = df["Discount_Amount"].sum()
total_net_sales = df["Net_Sales"].sum()
total_profit = df["Profit"].sum()
overall_margin = (total_profit / total_net_sales * 100) if total_net_sales else 0

print("Total orders:", len(df))
print("Total gross sales:", round(total_revenue, 2))
print("Total discounts:", round(total_discount, 2))
print("Total net sales:", round(total_net_sales, 2))
print("Total profit:", round(total_profit, 2))
print("Overall profit margin (%):", round(overall_margin, 2))
print("\\nHighest net-sales region:", region_sales.index[0])
print("Most profitable category:", category_profit.index[0])
print("Top-selling product:", top_products.index[-1])

## 9. Exporting the Cleaned Dataset

In [ ]:
output_path = "../output/retail_sales_cleaned.csv"
df.to_csv(output_path, index=False)
print("Cleaned dataset saved to:", output_path)

## 10. Project Conclusion

This project uses Python libraries to analyze retail sales and profitability. It explores revenue trends, product and category performance, regional sales, customer segments, discounts, profit margins, payment methods, and shipping duration.

The analysis demonstrates data cleaning, transformation, aggregation, statistical summaries, and visualization. The results apply only to the simulated educational dataset and should not be presented as actual company performance.

### Skills Demonstrated
- Data loading and exploration using Pandas
- Data cleaning and missing-value treatment
- Feature engineering using Pandas and NumPy
- Grouping, aggregation, and business metrics
- Data visualization using Matplotlib and Seaborn
- Exporting a cleaned dataset